# Minimal LOCC + LoRA XLM-R

Two-pass experiment:

1. Train `xlm-roberta-base` + LoRA normally on observed ratings.
2. Use that model as `P_theta(z | x)`, combine it with a cheap TF-IDF surface sentiment model, build LOCC posterior soft labels, then fine-tune a fresh LoRA model on those soft labels.

Note: by "xlm-bert" we mean the standard HF multilingual backbone `xlm-roberta-base`.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import inspect
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, Value
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, Trainer, TrainingArguments, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from experiments.config import ModelConfig
from experiments.models import SentimentModel

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_locc_lora_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128

BASE_EPOCHS = 2
LOCC_EPOCHS = 2
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 256
LR = 1.5e-4
FP16 = torch.cuda.is_available()

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data and tokenization

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")
df["lang"] = df.get("lang", "unk")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_hard(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [float(x) for x in batch["label"]]
    langs = batch["lang"] if "lang" in batch else ["unk"] * len(batch["sentence"])
    out["lang"] = [0 if x == "eng_Latn" else 1 for x in langs]
    return out

def to_hf_dataset(frame, tokenize_fn):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize_fn, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("labels", Value("float32"))
    if "lang" in ds.column_names:
        ds = ds.cast_column("lang", Value("int64"))
    ds.set_format("torch")
    return ds

train_ds = to_hf_dataset(train_df, tokenize_hard)
val_ds = to_hf_dataset(val_df, tokenize_hard)

## Helpers

In [ ]:
def make_model():
    cfg = ModelConfig(
        kind="lora_bert",
        name=MODEL_ID,
        geometry="default",
        lora_r=128,
        lora_alpha=64,
        lora_dropout=0.01,
    )
    model = SentimentModel.from_config(cfg)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.asarray(logits).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    rounded = np.rint(np.clip(preds, 0, 4))
    return {
        "mae": float(mean_absolute_error(labels, preds)),
        "rounded_mae": float(mean_absolute_error(labels, rounded)),
    }


def make_training_args(run_name, epochs):
    kwargs = dict(
        output_dir=str(OUTPUT_DIR / run_name / "checkpoints"),
        overwrite_output_dir=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=epochs,
        evaluation_strategy="steps",
        eval_steps=500,
        logging_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        seed=SEED,
    )
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)

## Pass 1: normal LoRA XLM-R

In [ ]:
base_model = make_model()
base_trainer = Trainer(
    model=base_model,
    args=make_training_args("base", BASE_EPOCHS),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
base_trainer.train()
base_metrics = base_trainer.evaluate()
base_metrics

## Build LOCC soft labels

In [ ]:
def scalar_to_class_dist(preds, temp=0.65):
    labels = np.arange(5, dtype=np.float32)[None, :]
    scores = -((preds.reshape(-1, 1) - labels) ** 2) / temp
    scores = scores - scores.max(axis=1, keepdims=True)
    probs = np.exp(scores)
    return probs / probs.sum(axis=1, keepdims=True)


def ordinal_kernel(observed, z, lam=1.35):
    labels = np.arange(5)
    return np.exp(-lam * np.abs(observed - z)) / np.exp(-lam * np.abs(labels - z)).sum()


def sarcasm_surface_kernel(v, z, alpha=1.20):
    labels = np.arange(5)
    opposite = 4 - z
    return np.exp(-alpha * np.abs(v - opposite)) / np.exp(-alpha * np.abs(labels - opposite)).sum()


def corruption_gate(y, v, pred, loss, pmax):
    pi = np.tile(np.array([0.72, 0.16, 0.06, 0.06], dtype=np.float32), (len(y), 1))
    d_yv = np.abs(y - v)
    d_yp = np.abs(y - pred)
    high_loss = loss > np.quantile(loss, 0.75)
    low_conf = pmax < np.quantile(pmax, 0.35)

    default0 = (y == 0) & (v >= 3) & (pred >= 3)
    sarcasm = (d_yv >= 3) & (pred == y)
    mislabel = high_loss & (d_yv >= 2) & (d_yp >= 2)
    clean = (d_yv <= 1) & (pred == y) & ~low_conf

    pi[default0, 2] += 1.50
    pi[sarcasm, 3] += 0.90
    pi[mislabel, 1] += 1.00
    pi[clean, 0] += 0.50
    return pi / pi.sum(axis=1, keepdims=True)


def locc_soft_labels(main_probs, y, v, eps=1e-12):
    pred = main_probs.argmax(axis=1)
    pmax = main_probs.max(axis=1)
    loss = -np.log(main_probs[np.arange(len(y)), y] + eps)
    pi = corruption_gate(y, v, pred, loss, pmax)
    q = np.zeros((len(y), 5), dtype=np.float32)

    for z in range(5):
        k_clean = (y == z).astype(np.float32)
        k_mis = ordinal_kernel(y, z).astype(np.float32)
        k_default0 = (y == 0).astype(np.float32)
        k_sarc = (y == z).astype(np.float32) * sarcasm_surface_kernel(v, z).astype(np.float32)
        channel = pi[:, 0] * k_clean + pi[:, 1] * k_mis + pi[:, 2] * k_default0 + pi[:, 3] * k_sarc
        q[:, z] = main_probs[:, z] * channel

    q = q + eps
    q = q / q.sum(axis=1, keepdims=True)
    diag = pd.DataFrame({
        "y": y,
        "v_surface": v,
        "pred_xlmr": pred,
        "loss": loss,
        "pmax": pmax,
        "pi_clean": pi[:, 0],
        "pi_mislabel": pi[:, 1],
        "pi_default0": pi[:, 2],
        "pi_sarcasm": pi[:, 3],
        "soft_argmax": q.argmax(axis=1),
        "soft_expected": q @ np.arange(5),
    })
    return q, diag

In [ ]:
# Cheap surface sentiment v = h(x).
surface_text_train = ("[lang=" + train_df["lang"].astype(str) + "] " + train_df["sentence"].astype(str)).to_numpy()
surface_text_val = ("[lang=" + val_df["lang"].astype(str) + "] " + val_df["sentence"].astype(str)).to_numpy()

surface_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=150_000, sublinear_tf=True)
X_surface_train = surface_vec.fit_transform(surface_text_train)
X_surface_val = surface_vec.transform(surface_text_val)
surface_clf = SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=8, tol=1e-3, n_jobs=-1, random_state=SEED)
surface_clf.fit(X_surface_train, train_df["label"].to_numpy())
v_train = surface_clf.predict(X_surface_train).astype(int)
v_val = surface_clf.predict(X_surface_val).astype(int)
print("surface val rounded MAE", mean_absolute_error(val_df["label"], v_val))

In [ ]:
train_raw = base_trainer.predict(train_ds).predictions.reshape(-1)
val_raw = base_trainer.predict(val_ds).predictions.reshape(-1)
print("base train rounded MAE", mean_absolute_error(train_df["label"], np.rint(np.clip(train_raw, 0, 4))))
print("base val rounded MAE", mean_absolute_error(val_df["label"], np.rint(np.clip(val_raw, 0, 4))))

main_probs_train = scalar_to_class_dist(np.clip(train_raw, 0, 4))
q_train, locc_diag = locc_soft_labels(main_probs_train, train_df["label"].to_numpy(dtype=int), v_train)

print("mean gate probabilities")
display(locc_diag[["pi_clean", "pi_mislabel", "pi_default0", "pi_sarcasm"]].mean().to_frame("mean"))
print("LOCC argmax changed fraction", (locc_diag["y"] != locc_diag["soft_argmax"]).mean())

interesting = locc_diag.assign(abs_shift=np.abs(locc_diag["soft_expected"] - locc_diag["y"])).sort_values("abs_shift", ascending=False).head(10)
display(train_df.loc[interesting.index, ["sentence", "label", "lang"]].join(interesting))

## Pass 2: train LoRA XLM-R on LOCC soft labels

In [ ]:
soft_train_df = train_df.copy()
soft_train_df["soft_labels"] = q_train.tolist()

def tokenize_soft(batch):
    out = tokenize_hard(batch)
    out["soft_labels"] = batch["soft_labels"]
    return out

soft_train_ds = to_hf_dataset(soft_train_df, tokenize_soft)

class LOCCSoftTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        soft_labels = inputs.pop("soft_labels").to(model.device).float()
        observed = inputs.pop("labels").to(model.device).float()
        outputs = model(**inputs)
        pred = outputs["logits"].view(-1).float()
        label_values = torch.arange(5, device=pred.device, dtype=pred.dtype).view(1, -1)

        soft_mae = (soft_labels * torch.abs(pred.view(-1, 1) - label_values)).sum(dim=1).mean()
        expected = (soft_labels * label_values).sum(dim=1)
        expected_huber = F.huber_loss(pred, expected, delta=0.75)
        anchor_huber = F.huber_loss(pred, observed, delta=1.0)
        loss = soft_mae + 0.25 * expected_huber + 0.10 * anchor_huber
        return (loss, outputs) if return_outputs else loss

In [ ]:
locc_model = make_model()
locc_trainer = LOCCSoftTrainer(
    model=locc_model,
    args=make_training_args("locc", LOCC_EPOCHS),
    train_dataset=soft_train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
locc_trainer.train()
locc_metrics = locc_trainer.evaluate()
print("base", base_metrics)
print("locc", locc_metrics)

In [ ]:
final_dir = OUTPUT_DIR / "final_locc_model"
locc_trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(final_dir)

## Optional submission from LOCC model

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")
    raw = locc_trainer.predict(test_ds).predictions.reshape(-1)
    preds = np.rint(np.clip(raw, 0, 4)).astype(int)
    submission = pd.DataFrame({"id": test_df["id"], "label": preds})
    submission_path = OUTPUT_DIR / "submission_locc.csv"
    submission_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(submission_path, index=False)
    print(submission_path)
    display(submission.head())